# 基于 YOLOv8 的脑部肿瘤 MRI 目标检测

## 1. 项目背景
本项目旨在利用深度学习模型（YOLOv8）对脑部 MRI 影像进行肿瘤目标的自动化检测与定位。
使用 YOLOv8m 模型进行训练，配置 Adamax 优化器以获得最佳检测性能。

## 2. 环境配置与依赖安装

In [ ]:
# !pip install ultralytics matplotlib opencv-python -q

## 3. 导入库与设备配置

In [ ]:
import os # 导入操作系统接口模块，用于处理文件路径和目录操作
import cv2 # 导入 OpenCV 库，用于读取图像和进行基本的图像处理操作
import torch # 导入 PyTorch 框架，作为深度学习模型训练的基础后端
import numpy as np # 导入 NumPy 库，用于高效的数组运算和随机数生成
import matplotlib.pyplot as plt # 导入 Matplotlib 绘图库，用于后续的数据可视化和结果展示
from ultralytics import YOLO # 导入 YOLO 类，这是 Ultralytics 提供的核心接口，用于加载、训练和推理模型
from PIL import Image # 导入 Python Imaging Library，用于兼容性地打开和显示图像

# 配置训练设备信息
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") # 自动检测是否有 GPU (cuda:0)，如果有则使用，否则回退到 CPU，确保代码可移植性
print(f"训练设备: {device}") # 打印当前选择的设备，方便用户确认硬件环境
if torch.cuda.is_available(): # 如果确认使用的是 GPU
    print(f"GPU: {torch.cuda.get_device_name(0)}") # 打印 GPU 的具体型号名称，让用户了解使用的算力硬件
    torch.backends.cudnn.benchmark = True # 开启 cuDNN 的自动寻优机制，它会为当前的输入尺寸寻找最快的卷积算法，虽然增加一点初始化时间，但能显著提升训练速度

## 4. 数据预览 (EDA)
可视化部分训练集样本及其标注框，包含对无肿瘤样本的处理。

In [ ]:
def visualize_samples(data_dir, num_samples=5):
    """
    从指定目录随机抽取样本图像，读取对应的 YOLO 标签文件，并在图像上绘制边界框。
    
    参数:
        data_dir (str): 包含 'images' 和 'labels' 子文件夹的数据集根目录路径
        num_samples (int): 希望随机展示的图片数量
    """
    img_dir = os.path.join(data_dir, 'images') # 拼接图像文件夹的绝对路径
    lbl_dir = os.path.join(data_dir, 'labels') # 拼接标签文件夹的绝对路径
    
    images = os.listdir(img_dir) # 获取该目录下所有的文件名列表
    sample_files = np.random.choice(images, num_samples, replace=False) # 从列表中随机不重复地选取 num_samples 个文件名
    
    fig, axes = plt.subplots(1, num_samples, figsize=(20, 5)) # 创建一个一行多列的画布，设置整体大小以便图片清晰展示
    
    for i, img_name in enumerate(sample_files): # 遍历选中的每一个文件名
        img_path = os.path.join(img_dir, img_name) # 获取当前图片的完整路径
        lbl_path = os.path.join(lbl_dir, img_name.replace('.jpg', '.txt')) # 根据图片名推导对应的标签 txt 文件路径
        
        img = cv2.imread(img_path) # 使用 OpenCV 读取图像数据 (默认 BGR 格式)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # 将颜色空间转换为 RGB，以便 Matplotlib 正确渲染颜色
        h, w, _ = img_rgb.shape # 获取图像的原始高度和宽度，用于后续的坐标反归一化
        
        has_tumor = False # 设置标志位，用于记录该图片是否包含肿瘤目标
        
        if os.path.exists(lbl_path): # 检查对应的标签文件是否存在
            with open(lbl_path, 'r') as f: # 打开标签文件
                lines = f.readlines() # 读取所有行，因为一张图可能有多个目标
            for line in lines: # 遍历每一行标签数据
                parts = list(map(float, line.strip().split())) # 将字符串按空格分割并转为浮点数列表
                if len(parts) == 5: # 校验 YOLO 格式数据完整性 (类别 x_center y_center w h)
                    has_tumor = True # 标记存在目标
                    
                    cls_id, cx, cy, bw, bh = parts # 解包归一化的中心点坐标和宽高
                    
                    x1 = int((cx - bw/2) * w) # 计算左上角 x 坐标：(中心 - 半宽) * 原图宽
                    y1 = int((cy - bh/2) * h) # 计算左上角 y 坐标：(中心 - 半高) * 原图高
                    x2 = int((cx + bw/2) * w) # 计算右下角 x 坐标：(中心 + 半宽) * 原图宽
                    y2 = int((cy + bh/2) * h) # 计算右下角 y 坐标：(中心 + 半高) * 原图高
                    
                    cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2) # 在图像上绘制绿色矩形框，线宽为 2
                    cv2.putText(img_rgb, 'Tumor', (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2) # 在框上方添加类别名称 'Tumor'
        
        if not has_tumor: # 如果标签不存在或为空，则判定为无肿瘤
            cv2.putText(img_rgb, 'No Tumor', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2) # 在图像左上角添加红色的 'No Tumor' 提示
            
        axes[i].imshow(img_rgb) # 在当前子图上显示绘制好的图像
        axes[i].axis('off') # 关闭坐标轴显示，保持图片纯净
        axes[i].set_title(img_name) # 设置子图标题为文件名
    
    plt.tight_layout() # 自动调整子图参数，防止图片重叠
    plt.show() # 渲染并弹出显示窗口

# 执行函数：展示训练集中随机抽取的 4 张图片
visualize_samples('/home/shao/CNNProject/Medical-DeepLearn/Torch/bigwork/3/dataset/train', num_samples=4)

## 5. 模型训练
使用 YOLOv8m 模型，配置 Adamax 优化器进行 50 轮训练。

In [ ]:
import gc # 导入垃圾回收模块，用于管理内存
# import ctypes

if 'model' in globals(): # 检查内存中是否已经存在名为 'model' 的变量
    del model # 删除旧模型对象，切断引用
if 'results' in globals():
    del results
gc.collect() # 强制触发垃圾回收机制，清理废弃的内存对象
if torch.cuda.is_available(): # 如果使用的是 GPU 环境
    torch.cuda.empty_cache() # 清空 PyTorch 在 GPU 上的显存缓存，防止显存碎片导致的内存溢出 (OOM)
# # 释放内存给OS
# libc = ctypes.CDLL("libc.so.6")
# libc.malloc_trim(0)


In [ ]:
# 加载模型配置：此处使用 yolov8m.pt 预训练权重文件
model = YOLO('yolov8m.pt') # 根据 v8m 配置文件构建模型架构

# 指定数据配置文件的路径，包含数据集目录、类别数等信息
data_yaml = '/home/shao/CNNProject/Medical-DeepLearn/Torch/bigwork/3/data.yaml' # 读取项目根目录下的 data.yaml

# 超参数配置
results = model.train(
    data=data_yaml, # 绑定数据集配置路径
    epochs=50, # 设定总训练轮次为 50 轮
    batch=16, # 设定批次大小为 16，需根据显卡显存调整，过大易导致 OOM
    optimizer='Adamax', # 选择 Adamax 优化器，通常具有较好的收敛稳定性
    imgsz=256, # 设定输入图像缩放的尺寸为 256x256 像素
    lr0 = 0.0005, # 设定初始学习率为 0.0005，较小的起步值有助于避免初始梯度爆炸
    lrf = 0.01, # 设定最终学习率为初始值的 0.01 倍，配合余弦退火策略逐渐减小步长
    momentum = 0.89, # 设定动量系数为 0.89，利用历史梯度方向加速收敛并减少震荡
    dropout = 0.4, # 设定 Dropout 概率为 0.4，强制网络随机丢弃 40% 的神经元以防止过拟合
    device=device, # 指定运行设备 (已在前面定义)
    project='runs', # 指定训练结果保存的父文件夹名称
    name='detect', # 指定本次实验的子文件夹名称，用于区分不同次的运行
    close_mosaic=10, # 设定在最后 10 轮关闭 Mosaic 数据增强，帮助模型在训练末期拟合细节
    val=True, # 开启验证模式，每轮训练结束后在验证集上评估性能
    plots=True, # 开启绘图模式，自动生成 Loss 和 mAP 的变化曲线图
    amp=False, # 关闭自动混合精度 (AMP)，在特定环境下避免精度不稳定导致的 NaN 错误
    workers=0, # 设定数据加载线程数为 0 (主进程加载)，防止多进程导致的环境死锁或崩溃
    cache=False, # 关闭数据缓存到内存，防止大批量图片加载时耗尽系统内存
    warmup_epochs=3, # 设定前 3 轮为预热期，学习率从 0 缓慢上升，保护预训练权重
    verbose=True, # 开启详细日志打印，在控制台实时输出训练进度
)

## 6. 模型评估与可视化
展示训练过程中的 Loss 曲线、mAP 曲线、混淆矩阵等。

In [ ]:
import os
import glob
from IPython.display import display, Image as IPyImage
# glob 模式中的末尾 '/' 确保只匹配文件夹，排除同名文件
runs = sorted(glob.glob('runs/detect*/'))
if runs:
    # 获取最新的训练结果目录
    latest_run = runs[-1]
    print(f"正在加载结果: {latest_run}")
    
    # 定义需要展示的文件
    files = {
        "Training Results (Loss & Metrics)": "results.png",
        "Confusion Matrix": "confusion_matrix.png",
        "Precision-Recall Curve": "PR_curve.png"
    }
    
    # 遍历并显示图片
    for title, filename in files.items():
        fpath = os.path.join(latest_run, filename)
        if os.path.exists(fpath):
            print(f"\n{title}:")
            display(IPyImage(fpath))
        else:
            print(f"未找到文件: {fpath}")
else:
    print("未找到 runs/detect* 目录，请确保训练已完成。")

## 7. 测试集推理与可视化
使用训练好的最佳模型在测试集上进行推理，并展示预测结果。

In [ ]:
import gc
# 清理训练阶段产生的显存占用
if 'model' in globals():
    del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

def visualize_predictions(model, test_img_dir, num_samples=5):
    images = os.listdir(test_img_dir)
    sample_files = np.random.choice(images, num_samples, replace=False)
    
    fig, axes = plt.subplots(1, num_samples, figsize=(20, 5))
    
    for i, img_name in enumerate(sample_files):
        img_path = os.path.join(test_img_dir, img_name)
        img = Image.open(img_path)
        
        # 推理
        results = model(img_path, verbose=False)
        
        # 绘制结果
        plotted = results[0].plot()
        plotted_img = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)
        
        axes[i].imshow(plotted_img)
        axes[i].axis('off')
        axes[i].set_title(img_name)
        
        # 检查是否有预测结果
        if len(results[0].boxes) == 0:
            axes[i].text(0.5, 0.5, 'No Tumor', ha='center', va='center', 
                        transform=axes[i].transAxes, color='red', fontsize=15, fontweight='bold')

    plt.tight_layout()
    plt.show()

## 训练时使用，自动获取最新训练记录的最佳模型
# # 加载最佳模型
# best_model_path = 'runs/detect/weights/best.pt' 
# # 这里使用最新运行的 best.pt
# import glob
# weights_paths = glob.glob('runs/detect*/weights/best.pt')
# if weights_paths:
#     best_model_path = sorted(weights_paths)[-1]
#     print(f"Loading model from: {best_model_path}")
#     best_model = YOLO(best_model_path)
#     visualize_predictions(best_model, '/home/shao/CNNProject/Medical-DeepLearn/Torch/bigwork/3/dataset/test/images', num_samples=5)
# else:
#     print("Model not trained yet.")

best_model_path = 'runs/r93-p63/weights/best.pt' 
print(f"Loading model from: {best_model_path}")
best_model = YOLO(best_model_path)
visualize_predictions(best_model, '/home/shao/CNNProject/Medical-DeepLearn/Torch/bigwork/3/dataset/test/images', num_samples=5)

## 8. 模型保存
将最佳模型权重复制到项目根目录以便使用。

In [ ]:
import gc
import torch

# 清理显存，释放之前步骤可能占用的资源
if 'model' in globals(): # 检查全局变量中是否存在 'model'
    del model # 如果存在，则删除以释放引用
gc.collect() # 强制垃圾回收清理 Python 对象
if torch.cuda.is_available(): # 如果使用 GPU
    torch.cuda.empty_cache() # 清空 GPU 显存缓存

In [ ]:
import os # 导入操作系统模块
import gc # 导入垃圾回收模块
import cv2 # 导入 OpenCV 库用于图像处理
import glob # 导入 glob 模块用于路径匹配查找文件
import shutil # 导入 shutil 模块用于文件复制操作
import torch # 导入 PyTorch 框架
import numpy as np # 导入 NumPy 库
import matplotlib.pyplot as plt # 导入 Matplotlib 绘图库
from ultralytics import YOLO # 导入 YOLO 模型类
from PIL import Image # 导入 PIL 图像处理库


# 获取真实标签的辅助函数
def get_ground_truth(img_name, labels_dir):
    """
    读取图像对应的标签文件，解析出真实的类别信息。
    
    参数:
        img_name (str): 图像文件名 (如 'img.jpg')
        labels_dir (str): 标签文件所在的目录路径
        
    返回:
        str: 标签内容 ('Tumor', 'No Tumor', 'Unknown', 或 'Error')
    """
    base_name = os.path.splitext(img_name)[0] # 分离文件名和后缀
    txt_path = os.path.join(labels_dir, base_name + ".txt") # 拼接 txt 标签路径
    if not os.path.exists(txt_path): # 检查文件是否存在
        return "Unknown" # 不存在则返回 Unknown
    try: # 尝试读取文件
        with open(txt_path, 'r') as f: # 打开标签文件
            content = f.read().strip() # 读取内容并去除空白
        return "Tumor" if content else "No Tumor" # 有内容返回 Tumor，否则返回 No Tumor
    except: # 捕获读取异常
        return "Error" # 出错返回 Error

# 九宫格可视化函数
def visualize_nine_grid(model, test_img_dir, test_lbl_dir, num_samples=9):
    """
    在 3x3 的网格中展示测试集图像的预测结果，并对比真实标签。
    
    参数:
        model: 训练好的 YOLO 模型对象
        test_img_dir (str): 测试集图像目录
        test_lbl_dir (str): 测试集标签目录
        num_samples (int): 展示的样本数量 (默认 9)
    """
    images = [f for f in os.listdir(test_img_dir) if f.endswith('.jpg')] # 筛选出所有 jpg 格式的图像
    actual_samples = min(num_samples, len(images)) # 确保请求数量不超过实际图片总数
    
    fig, axes = plt.subplots(3, 3, figsize=(15, 15)) # 创建 3x3 的子图网格，设置整体大小
    axes = axes.flatten() # 将二维的 axes 数组展平为一维，方便遍历
    
    for i in range(actual_samples, 9): # 如果图片不足 9 张，隐藏多余的子图
        axes[i].axis('off') # 关闭空子图的坐标轴

    sample_files = np.random.choice(images, actual_samples, replace=False) # 随机不重复抽取图片
    
    for i, img_name in enumerate(sample_files): # 遍历抽取的图片
        img_path = os.path.join(test_img_dir, img_name) # 图像路径
        gt_label = get_ground_truth(img_name, test_lbl_dir) # 获取真实标签
        
        results = model(img_path, verbose=False) # 进行推理，verbose=False 减少控制台输出
        plotted_img = results[0].plot() # 获取带有预测框的图像 (BGR 格式)
        
        if len(results[0].boxes) > 0: # 如果模型检测到了目标
            pred_conf = results[0].boxes.conf[0].item() # 获取第一个目标的置信度
            pred_label = "Tumor" # 判定为 Tumor
            pred_info = f"Pred: {pred_label} ({pred_conf:.2f})" # 格式化预测信息
        else: # 如果没检测到
            pred_label = "No Tumor" # 判定为 No Tumor
            pred_info = f"Pred: {pred_label}" # 格式化预测信息
            
        is_correct = (gt_label == pred_label) # 比较预测与真实标签是否一致
        title_color = 'green' if is_correct else 'red' # 正确显示绿色，错误显示红色
        
        img_rgb = cv2.cvtColor(plotted_img, cv2.COLOR_BGR2RGB) # 转换颜色格式以正确显示
        axes[i].imshow(img_rgb) # 显示图像
        axes[i].axis('off') # 关闭坐标轴
        
        axes[i].set_title(f"GT: {gt_label}\n{pred_info}", color=title_color, fontsize=12, pad=10) # 设置标题，包含 GT 和 Pred 信息

    plt.tight_layout() # 调整布局
    plt.show() # 显示图像

load_path = './runs/r93-p63/weights/best.pt' # 模型路径：可自定义路径 
# 执行
print(f"Loading model from: {load_path}") # 打印加载路径
best_model = YOLO(load_path) # 加载模型

test_img_dir = '/home/shao/CNNProject/Medical-DeepLearn/Torch/bigwork/3/dataset/test/images' # 测试图像路径
test_lbl_dir = '/home/shao/CNNProject/Medical-DeepLearn/Torch/bigwork/3/dataset/test/labels' # 测试标签路径

visualize_nine_grid(best_model, test_img_dir, test_lbl_dir, num_samples=9) # 执行九宫格可视化